# Train PatchCore anomaly detector on Kaggle (MVTec AD)

Clones the actual repo (private) using a GitHub token stored as a
Kaggle Secret, then runs the same `training/train_patchcore.py` and
`training/evaluate_patchcore.py` scripts used locally — no duplicated
logic here, same pattern as `train_yolo_kaggle.ipynb`.

Unlike NEU-DET, **MVTec AD needs no manual download or Kaggle Input** —
anomalib fetches the category archive directly from the official MVTec
mirror at train time, given internet access.

Before running:

1. **Add-ons > Secrets**: check `GITHUB_TOKEN` ON for this notebook.
2. **Settings > Accelerator**: GPU (T4 x2 or P100).
3. **Settings > Internet**: On (needed for `pip install anomalib` and
   the MVTec AD download itself).

Then run all cells top to bottom.

## 1. Clone the repo using the GitHub token secret

Same safeguards as the YOLO notebook: the token is never printed, and
the clone URL (which briefly embeds it) is immediately overwritten in
`.git/config` right after cloning.

In [ ]:
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_OWNER_REPO = 'satyazm/factory-defect-detection'
CLONE_DIR = '/kaggle/working/repo'

token = UserSecretsClient().get_secret('GITHUB_TOKEN')
auth_url = f'https://{token}@github.com/{GITHUB_OWNER_REPO}.git'
clean_url = f'https://github.com/{GITHUB_OWNER_REPO}.git'

result = subprocess.run(
    ['git', 'clone', '--depth', '1', auth_url, CLONE_DIR],
    capture_output=True, text=True,
)
if result.returncode != 0:
    sanitized = result.stderr.replace(token, '***')
    raise RuntimeError(f'git clone failed:\n{sanitized}')

subprocess.run(['git', '-C', CLONE_DIR, 'remote', 'set-url', 'origin', clean_url], check=True)
del token, auth_url

print('Cloned to', CLONE_DIR)


In [ ]:
%cd /kaggle/working/repo


## 2. Install anomalib

This can take a few minutes — it may need to adjust `torch`/`lightning`
versions to match what it requires (>=2.6 / >=2.2).

In [ ]:
!pip install -q anomalib


## 3. Pick a category and train

Any of the 15 MVTec AD categories works — `bottle` matches the
conveyor-belt bottle-inspection framing from the project writeup.
Change `CATEGORY` and re-run from here to train a different one.

PatchCore trains only on "good" images and builds a coreset feature
memory bank in a single pass (no iterative loss to converge), so this
is fast even relative to the dataset download itself.

In [ ]:
CATEGORY = 'bottle'


In [ ]:
!python training/train_patchcore.py --category {CATEGORY}


## 4. Evaluate (optional)

Reports image-level AUROC and F1 on the full test split (good + every defect type for this category).

In [ ]:
!python training/evaluate_patchcore.py --category {CATEGORY}


## 5. Get the trained weights back to your local repo

The exported model lands at
`models/patchcore/<category>/weights/torch/model.pt`. Zip it below,
then use the notebook's **Output** pane (after *Save Version → Save &
Run All*) to download it. Copy the extracted `model.pt` into your local
repo at `models/patchcore/<category>/weights/torch/model.pt` — it's
gitignored, so this is a manual file copy, not a git operation.

In [ ]:
import shutil

weights_dir = f'/kaggle/working/repo/models/patchcore/{CATEGORY}/weights/torch'
shutil.make_archive(f'/kaggle/working/patchcore_{CATEGORY}_weights', 'zip', weights_dir)
print(f'Zipped weights at /kaggle/working/patchcore_{CATEGORY}_weights.zip')
